# Gráfico de quadrantes — perfis de vocabulário dos planos nacionais de inteligência artificial

Este notebook reproduz integralmente a figura `figura.png`: da leitura dos documentos-fonte
até a exportação do gráfico em PNG (300 dpi) e PDF. Ele foi escrito para ser **aberto,
executado e modificado por terceiros**: todos os parâmetros de análise e de desenho estão
expostos em células próprias, e o cálculo e o desenho ficam em dois módulos Python na mesma
pasta (`calcular.py` e `plotar.py`), que são a única fonte da verdade — o notebook os importa
em vez de duplicar o código, de modo que a figura gerada aqui e a gerada pela linha de comando
são idênticas por construção.

## Corpus

| Documento | País/bloco | Ano | Arquivo-fonte |
|---|---|---|---|
| AI Continent Action Plan | União Europeia | 2025 | `ai_continent_action_plan.json` |
| Apply AI Strategy | União Europeia | 2025 | `apply_ai_strategy.json` |
| America's AI Action Plan | Estados Unidos | 2025 | `americas_ai_action_plan.json` |
| New Generation AI Development Plan | China | 2017 | `new_generation_ai_development_plan.json` |
| Opinions on the "Artificial Intelligence+" Initiative | China | 2025 | `ai_plus.json` |
| Plano Brasileiro de Inteligência Artificial (PBIA) | Brasil | 2024 | `pbia.json` |

Os seis arquivos JSON estão na pasta imediatamente acima desta (`notebooks/ Novo/`) e contêm o
texto integral de cada documento, já limpo de capas, sumários e numeração de página.

## O que este notebook produz

| Arquivo | Conteúdo |
|---|---|
| `figura.png` / `figura.pdf` | o gráfico de quadrantes (300 dpi e vetorial) |
| `coordenadas.csv` | documento, país, ano, G2, G3, G4, G5, massa, X, Y |
| `robustez.csv` | as três versões de coordenadas lado a lado e a indicação de mudança de quadrante |
| `termos_ausentes.md` | termos do codebook não encontrados em cada documento, e demais diagnósticos |

## Requisitos

Python 3.10 ou superior com `pandas`, `matplotlib` e — apenas se `FONTE = "json"` (ver adiante) —
`spacy` com o modelo `en_core_web_sm`:

```bash
pip install pandas matplotlib spacy
python -m spacy download en_core_web_sm
```

## Como executar

Abra este notebook **a partir da própria pasta `quadrantes/`** e execute as células na ordem
(menu *Run* → *Run All*). A execução completa leva de um a três minutos quando parte dos JSONs,
e poucos segundos quando parte dos CSVs já exportados.

---
## 1. Preparação do ambiente

A célula abaixo localiza a pasta de trabalho, importa os dois módulos e os **recarrega**: se você
editar `calcular.py` ou `plotar.py` com o notebook aberto, basta executar esta célula de novo
para que as mudanças passem a valer.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# A pasta do notebook é a pasta 'quadrantes'; os documentos-fonte estão um nível acima.
DIR_NOTEBOOK = Path.cwd()
if not (DIR_NOTEBOOK / "calcular.py").exists():
    raise FileNotFoundError(
        "Execute este notebook a partir da pasta 'quadrantes' (onde estão calcular.py e "
        f"plotar.py). Pasta atual: {DIR_NOTEBOOK}")
DIR_DADOS = DIR_NOTEBOOK.parent

if str(DIR_NOTEBOOK) not in sys.path:
    sys.path.insert(0, str(DIR_NOTEBOOK))

import calcular
import plotar

importlib.reload(calcular)   # garante que edições nos módulos entrem em vigor
importlib.reload(plotar)

%matplotlib inline
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

print("Pasta do notebook :", DIR_NOTEBOOK)
print("Pasta dos dados   :", DIR_DADOS)
print("Documentos (JSON) :", len(sorted(DIR_DADOS.glob("*.json"))), "arquivos encontrados")

---
## 2. Parâmetros da análise

Três decisões controlam todo o resto. **Altere aqui e reexecute o notebook** para obter uma
variante da análise.

- **`FONTE`** — de onde vêm as tabelas de frequência de cada documento.
  - `"json"` regenera as tabelas a partir dos seis documentos originais, reexecutando o mesmo
    pipeline linguístico do notebook `analise_lexical_planos_ia.ipynb` (spaCy, lematização com
    reconhecimento de classe gramatical, *stopwords* por documento). É a opção reprodutível
    ponta a ponta e a única que permite `TOP_N` diferente de 50.
  - `"csv"` lê os arquivos `*_top50_vocabulario.csv` já exportados por aquele notebook. Não
    exige spaCy e roda em segundos; serve só para o top-50.
- **`TOP_N`** — quantos termos de cada documento entram na conta: `50` (versão atual da
  pesquisa), `100`, ou `0` para o vocabulário inteiro do documento.
- **`REGRA_BIGRAMAS`** — como um bigrama é atribuído a um grupo. Afeta **apenas** a versão de
  robustez que inclui bigramas; a versão principal usa somente unigramas.
  - `"nucleo"`: o bigrama herda o grupo do seu núcleo (última palavra) — *Public Sector* → G3.
  - `"dividir"`: a taxa é repartida entre os grupos das duas palavras — *Public Sector* → ½ G4, ½ G3.

Com `FONTE = "json"` e `TOP_N = 50`, o notebook confere automaticamente, termo a termo, se as
tabelas regeneradas coincidem com os CSVs exportados pelo notebook de análise lexical.

In [ ]:
FONTE = "json"            # "json" (reprodução completa, usa spaCy) ou "csv" (rápido, só top-50)
TOP_N = 50                # 50, 100, ou 0 para o documento inteiro
REGRA_BIGRAMAS = "nucleo" # "nucleo" ou "dividir"

print(f"Fonte das tabelas .....: {FONTE}")
print(f"Termos por documento ..: {'todos' if TOP_N == 0 else f'top-{TOP_N}'}")
print(f"Regra para bigramas ...: {REGRA_BIGRAMAS}")

---
## 3. O codebook

O codebook classifica cada lema do vocabulário em cinco grupos. O **G1 é descartado**: reúne o
vocabulário genérico de inteligência artificial e os verbos de política pública (*develop*,
*promote*, *enhance*), que aparecem em todos os documentos e não os diferenciam. Os quatro
grupos restantes formam os polos do gráfico:

| Grupo | Polo na figura | Conteúdo |
|---|---|---|
| G2 | **A**, topo | infraestrutura, computação e base material |
| G3 | **D**, direita | economia, indústria, difusão de mercado e competição geopolítica |
| G4 | **B**, esquerda | Estado, governança e regulação |
| G5 | **C**, base | ciência, dimensão social, capital humano, inclusão e sustentabilidade |

Os pares que se opõem em cada eixo são conceituais, não arbitrários: o eixo horizontal contrapõe
a lógica econômico-competitiva (G3) à lógica institucional-regulatória (G4) — *quem conduz*; o
eixo vertical contrapõe a base material (G2) à capacidade humana e científica (G5) — *em que se
investe*.

**Regra de casamento.** A comparação é por **igualdade exata do lema**, sem distinção de
maiúsculas — nunca por prefixo ou por trecho de palavra. Assim, *Act* não captura *Action*,
*Open* não captura *Open-source* e *Center* não captura *Centre* (ambos estão listados
separadamente). Variantes ortográficas (`VARIANTES`, p. ex. *datum* → *data*, *defense* →
*defence*) e plurais são reconhecidos e **ficam registrados** em `termos_ausentes.md`, para
conferência. Um mesmo lema em dois grupos interrompe a execução com erro.

### Como alterar o codebook

Há dois caminhos, ambos legítimos:

1. **Permanente** — edite o dicionário `CODEBOOK` no início de `calcular.py` e reexecute a
   célula da seção 1 (que recarrega o módulo).
2. **Experimental, só nesta sessão** — altere a lista em memória e reconstrua o índice, como no
   exemplo comentado na célula abaixo. Nada é gravado em `calcular.py`.

Em qualquer dos casos, depois de mudar o codebook confira a seção 9 (`termos_ausentes.md`): ela
lista os termos do top-N que ficaram **fora** de todos os grupos, que são exatamente os
candidatos a entrar numa próxima revisão.

In [ ]:
for grupo, termos in calcular.CODEBOOK.items():
    marca = "  (descartado)" if grupo == "G1" else ""
    print(f"{grupo} — {calcular.NOMES_GRUPOS[grupo]}{marca}")
    print(f"   {len(termos)} termos: " + ", ".join(termos) + "\n")

# --- Exemplo de alteração só nesta sessão (descomente para usar) ---------------
# calcular.CODEBOOK["G3"].append("Procurement")   # acrescenta um termo a G3
# calcular.CODEBOOK["G5"].remove("Service")       # remove um termo de G5
# calcular.recarregar_codebook()                  # OBRIGATÓRIO após qualquer mudança

---
## 4. Cálculo das coordenadas

Para cada documento, o peso de um grupo é a **soma das taxas por 1.000 tokens** dos seus termos
(nunca a contagem bruta: os documentos vão de cerca de 2,9 mil a 17,2 mil tokens). Um termo do
codebook ausente do documento vale zero. As coordenadas são índices de balanço entre polos
opostos, sempre no intervalo [−1, +1], iguais a zero no equilíbrio:

$$X = \frac{G3 - G4}{G3 + G4} \qquad\qquad Y = \frac{G2 - G5}{G2 + G5}$$

O tamanho da bolha é a **massa temática total**, G2 + G3 + G4 + G5.

A célula abaixo executa a análise e grava `coordenadas.csv`, `robustez.csv` e
`termos_ausentes.md` nesta pasta.

In [ ]:
tabelas, resultados, coordenadas, robustez = calcular.executar(
    dir_dados=DIR_DADOS,
    fonte=FONTE,
    top=TOP_N,
    bigramas=REGRA_BIGRAMAS,
    saida=DIR_NOTEBOOK,
)

In [ ]:
coordenadas[["documento", "país", "ano", "G2", "G3", "G4", "G5", "massa", "X", "Y"]].round(2)

---
## 5. Conferência com os valores de referência

Três documentos têm coordenadas de referência conhecidas, calculadas de forma independente a
partir do top-50 (versão soma, só unigramas). A tabela abaixo compara os valores obtidos com
esses pontos de controle; a tolerância é de ±0,01, que é o limite do arredondamento das taxas.

Se você mudou `TOP_N` ou o codebook, é **esperado** que os valores difiram — outra base de
termos produz outras somas. Nesse caso a coluna de conferência deixa de ser um teste de
correção e passa a medir o efeito da mudança.

In [ ]:
linhas = []
for arquivo, (x_ref, y_ref) in calcular.REFERENCIA.items():
    linha = coordenadas.loc[coordenadas["arquivo"] == arquivo].iloc[0]
    confere = abs(linha["X"] - x_ref) < 0.01 and abs(linha["Y"] - y_ref) < 0.01
    linhas.append({
        "documento": linha["rotulo"],
        "X calculado": round(float(linha["X"]), 4), "X referência": x_ref,
        "Y calculado": round(float(linha["Y"]), 4), "Y referência": y_ref,
        "confere (±0,01)": "sim" if confere else "não",
    })

conferencia = pd.DataFrame(linhas)
if TOP_N != 50:
    print(f"Atenção: a base atual é top-{TOP_N or 'integral'}, não o top-50 das referências.\n")
conferencia

---
## 6. A figura

Esta é a figura do arquivo `figura.png`. Ela é montada por `plotar.construir_figura()`, que
devolve também a lista de **avisos**: o código mede, em pixels, se algum rótulo colidiu com
outro rótulo, com uma bolha que não é a sua, ou se dois blocos de texto (título, subtítulo,
legenda, parágrafo explicativo, nota) se sobrepuseram. Lista vazia significa que o desenho saiu
limpo — vale a pena conferir sempre que os dados ou os textos mudarem.

**Cores.** Cada país ou bloco tem uma cor fixa, definida em `plotar.CORES_PAIS`. A paleta foi
verificada para as três formas principais de daltonismo: a separação mínima entre pares é
ΔE 9,8 em deuteranopia e 11,2 em tritanopia (espaço OKLab), acima do piso de 8 recomendado.
Ainda assim, **nenhuma informação depende só da cor**: cada bolha tem o rótulo ao lado, a
legenda repete país, documento e ano, e o PBIA — documento em foco da pesquisa — recebe anel
tracejado e rótulo em negrito.

Para alterar as cores, os textos ou o espaçamento dos blocos, edite as constantes no início de
`plotar.py` (`CORES_PAIS`, `TITULO`, `SUBTITULO`, `COMO_LER`, `POLOS`, `OFFSETS`, `FIGSIZE`) e
reexecute a célula da seção 1.

In [ ]:
df_figura = plotar.carregar_coordenadas(DIR_NOTEBOOK, versao="soma_uni")
fig, ax, avisos = plotar.construir_figura(df_figura)

if avisos:
    for aviso in avisos:
        print("AVISO:", aviso)
else:
    print("Conferência automática: nenhuma sobreposição de rótulos, bolhas ou blocos de texto.")

display(fig)
plt.close(fig)   # evita que o backend inline exiba a mesma figura duas vezes

### Exportação

`figura.png` em 300 dpi para impressão e `figura.pdf` vetorial, com as fontes embutidas como
texto (`pdf.fonttype = 42`), de modo que o PDF permaneça pesquisável e editável em editores
vetoriais.

In [ ]:
png, pdf = plotar.salvar(fig, DIR_NOTEBOOK, "figura")
print("Gravados:")
print("  ", png)
print("  ", pdf)

---
## 7. Robustez

A posição de um documento não deveria depender de escolhas arbitrárias do método. Duas
alternativas são calculadas junto com a versão principal:

1. **`media_uni`** — o peso de cada grupo é dividido pelo número de termos que o compõem no
   codebook (média em vez de soma). Isso neutraliza o fato de que G3 tem 33 termos e G4 apenas
   11; como G4 é o menor grupo, essa versão desloca sistematicamente o eixo X para a esquerda.
   É uma propriedade conhecida da normalização, não um erro.
2. **`soma_uni_bi`** — inclui também os bigramas do top-N, atribuídos pela regra escolhida em
   `REGRA_BIGRAMAS`.

A coluna `muda_quadrante` indica os documentos que trocam de quadrante entre as três versões.

In [ ]:
colunas = (["documento"]
           + [f"{v}_{c}" for v in ("soma_uni", "media_uni", "soma_uni_bi") for c in ("X", "Y")]
           + [f"{v}_quadrante" for v in ("soma_uni", "media_uni", "soma_uni_bi")]
           + ["muda_quadrante"])
robustez[colunas].round(3)

As duas versões alternativas também podem ser desenhadas, com o mesmo código e a mesma paleta.
O rodapé de cada figura registra qual definição de peso foi usada.

In [ ]:
for versao in ("media_uni", "soma_uni_bi"):
    df_v = plotar.carregar_coordenadas(DIR_NOTEBOOK, versao=versao)
    fig_v, _, avisos_v = plotar.construir_figura(
        df_v, versao=versao,
        titulo=f"{plotar.TITULO} — versão de robustez “{versao}”")
    display(fig_v)
    plt.close(fig_v)
    for aviso in avisos_v:
        print(f"AVISO ({versao}):", aviso)

---
## 8. Termos ausentes e diagnósticos

`termos_ausentes.md` registra, documento a documento: os termos de cada grupo **presentes** (com
a respectiva taxa) e os **ausentes**; a massa descartada em G1; os unigramas do top-N que
ficaram **fora do codebook** (candidatos a uma próxima revisão da classificação); os casamentos
aceitos por variante ortográfica ou plural; e a classificação de cada bigrama. É o arquivo a
consultar antes de alterar a classificação de um termo.

In [ ]:
texto = (DIR_NOTEBOOK / "termos_ausentes.md").read_text(encoding="utf-8")
print(f"termos_ausentes.md — {len(texto.splitlines())} linhas. Trecho inicial:\n")
display(Markdown("\n".join(texto.splitlines()[:26])))

---
## 9. Como adaptar este notebook

| Objetivo | Onde mexer |
|---|---|
| Incluir ou reclassificar um termo | `CODEBOOK`, no início de `calcular.py` (ou em memória, seção 3) |
| Expandir para o top-100 ou para o documento inteiro | `TOP_N`, na seção 2 |
| Acrescentar um documento ao corpus | `DOCUMENTOS` em `calcular.py` (+ o JSON na pasta de dados) e, se for um novo país, `CORES_PAIS` em `plotar.py` |
| Trocar cores, título, subtítulo ou legenda | constantes no início de `plotar.py` |
| Ajustar a posição de um rótulo que colidiu | `OFFSETS` em `plotar.py` (a figura avisa quando há colisão) |
| Gerar a figura fora do notebook | `python calcular.py && python plotar.py` na pasta `quadrantes` |

### Limitações a declarar em qualquer uso acadêmico

- Dois documentos do corpus (`ai_plus.json` e `new_generation_ai_development_plan.json`) são
  **traduções** do chinês para o inglês; o vocabulário medido reflete também escolhas dos
  tradutores.
- O codebook é um instrumento **interpretativo**: a alocação de termos ambíguos (*security*,
  *control*, *service*) a um grupo é uma decisão teórica, explicitada no código justamente para
  poder ser contestada e alterada por quem replicar a análise.
- A base atual é o top-50 de cada documento. Ampliar a base aproxima os documentos do centro do
  gráfico, porque termos menos frequentes distribuem massa por mais grupos; as três versões de
  `robustez.csv` permitem medir esse efeito.

---
## 10. Versão com estética revisada

As figuras desta seção **não substituem** a da seção 6 — `figura.png`, `figura.pdf` e `plotar.py`
continuam como estavam. São a mesma figura: mesmos dados (`coordenadas.csv`), mesmos eixos e
limites, mesmas linhas de referência, mesmas caixas dos quatro vocabulários, mesmos rótulos e
deslocamentos. Só a estética muda, pelo módulo `plotar_v2.py`, que importa `plotar.py` sem
alterá-lo:

- **sem o parágrafo "Como ler o gráfico"**, o que libera espaço para um gráfico maior (lado de
  7,4 pol. em vez de 6,9);
- **legenda de documentos junto do gráfico**, com a **massa temática no mesmo painel**;
- **cores por país ou bloco**: Estados Unidos em azul-escuro, China em vermelho, União Europeia
  em amarelo e Brasil em verde (`plotar_v2.CORES_PAIS`).

Os tons foram ajustados para que o vermelho e o verde continuem distinguíveis em daltonismo:
a separação mínima entre qualquer par de cores é ΔE 11,5 em deuteranopia e protanopia e 22,2 em
tritanopia (espaço OKLab). O amarelo tem pouco contraste contra o fundo branco; por isso cada
bolha tem borda escura e rótulo ao lado, e o PBIA mantém o anel tracejado e o negrito.

Além das conferências de sobreposição da seção 6, `plotar_v2.construir_figura` lê de volta, da
própria figura, a posição de cada bolha desenhada e confere que ela é **idêntica** a X e Y de
`coordenadas.csv`. Os círculos da legenda de massa temática têm a mesma escala das bolhas.

São dois arranjos da legenda, para escolher o que se encaixa melhor na página:

### 10.1 Legenda no canto superior esquerdo

O painel ocupa o quadrante superior esquerdo, que não tem nenhum documento nesta versão dos
dados; seu topo coincide com a borda superior do gráfico e sua margem esquerda com a da caixa do
Vocabulário B. Grava `figura_v2_quadrante.png` (300 dpi) e `figura_v2_quadrante.pdf`.

In [ ]:
import plotar_v2
importlib.reload(plotar_v2)   # garante que edições em plotar_v2.py entrem em vigor

df_v2 = plotar.carregar_coordenadas(DIR_NOTEBOOK, versao="soma_uni")
fig_v2, ax_v2, avisos_v2 = plotar_v2.construir_figura(df_v2, layout="quadrante")

if avisos_v2:
    for aviso in avisos_v2:
        print("AVISO:", aviso)
else:
    print("Conferência automática: posições idênticas às de coordenadas.csv; nenhuma sobreposição "
          "de rótulos, bolhas, painel de legenda ou blocos de texto.")

png, pdf = plotar_v2.salvar(fig_v2, DIR_NOTEBOOK, "figura_v2_quadrante")
print("Gravados:", png.name, "e", pdf.name)
display(fig_v2)
plt.close(fig_v2)

### 10.2 Legenda abaixo do gráfico, com a massa temática ao lado

Alternativa para quando o quadrante superior esquerdo não estiver vazio — por exemplo, na versão
de robustez `media_uni`, em que os EUA passam para X < 0. O painel fica centralizado sob a caixa
do Vocabulário C, com os documentos à esquerda e a massa temática à direita. Grava
`figura_v2_abaixo.png` e `figura_v2_abaixo.pdf`.

In [ ]:
fig_v2b, ax_v2b, avisos_v2b = plotar_v2.construir_figura(df_v2, layout="abaixo")

if avisos_v2b:
    for aviso in avisos_v2b:
        print("AVISO:", aviso)
else:
    print("Conferência automática: posições idênticas às de coordenadas.csv; nenhuma sobreposição "
          "de rótulos, bolhas, painel de legenda ou blocos de texto.")

png, pdf = plotar_v2.salvar(fig_v2b, DIR_NOTEBOOK, "figura_v2_abaixo")
print("Gravados:", png.name, "e", pdf.name)
display(fig_v2b)
plt.close(fig_v2b)